In this document we train a language model using Elman RNN (vanilla RNN) on a tiny English corpus.

The goal is to learn

$$
p(w_{t+1}| w_{1:t})
$$

from a short text using next-token prediction. The key equations are described in the slides:

$$
\begin{array}{rcl}
h_t & = & \text{tanh}(W_{hh} h_{t-1} + W_{ih} x_t + b_h)\\
o_t & = & W_{ho} h_t + b_o\\
p_t & = & \text{softmax}(o_t)
\end{array}
$$

## Modules

In [ ]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 66.4 MB/s eta 0:00:00


In [ ]:

from __future__ import annotations
import random
from dataclasses import dataclass
from typing import List, Tuple, Dict, Optional

import gensim

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

## Data Preparation

In [ ]:

def load_spacy(model_name: str = "en_core_web_sm"):
    """
    Load a spaCy model. Raises a helpful error if unavailable.
    """
    try:
        import spacy  # type: ignore
        return spacy.load(model_name)
    except Exception as e:
        msg = (
            "Could not load spaCy model.\n"
            "Make sure you have installed spaCy and the language model, e.g.\n"
            "  pip install spacy\n"
            "  python -m spacy download en_core_web_sm\n"
            f"Original error: {e}"
        )
        raise RuntimeError(msg) from e


def spacy_tokenize_paragraphs(nlp, text: str) -> List[List[str]]:
    """
    Tokenize text into sentences, returning a list of token lists.

    We keep only alphabetic tokens and normalize to lowercase lemmas.
    This avoids exploding the vocabulary for tiny corpora.

    Returns:
      sentences_tokens: list of sentences, each a list of tokens
    """
    try:
        doc = nlp(text)
        sentences_tokens: List[List[str]] = []

        for sent in doc.sents:
            toks: List[str] = []
            for tok in sent:
                if tok.is_space:
                    continue
                if tok.is_punct:
                    continue
                if not tok.is_alpha:
                    continue
                lemma = tok.lemma_.lower().strip()
                if len(lemma) == 0:
                    continue
                toks.append(lemma)

            if len(toks) >= 2:
                sentences_tokens.append(toks)

        if len(sentences_tokens) == 0:
            raise ValueError("No usable sentences found after filtering. Provide more text.")

        return sentences_tokens
    except Exception as e:
        raise RuntimeError(f"spaCy tokenization failed: {e}") from e


## Word2Vec Encoding

In [ ]:
def train_word2vec(
    sentences_tokens: List[List[str]],
    vector_size: int = 50,
    window: int = 5,
    min_count: int = 1,
    epochs: int = 50,
    seed: int = 0,
):
    """
    Train a Word2Vec model on tokenized sentences.
    """
    try:
        from gensim.models import Word2Vec

        model = Word2Vec(
            sentences=sentences_tokens,
            vector_size=vector_size,
            window=window,
            min_count=min_count,
            sg=1,
            workers=1,
            seed=seed,
            epochs=epochs,
        )
        return model
    except Exception as e:
        msg = (
            "Could not train Word2Vec. Make sure gensim is installed:\n"
            "  pip install gensim\n"
            f"Original error: {e}"
        )
        raise RuntimeError(msg) from e


def build_vocab_from_w2v(w2v_model) -> Tuple[Dict[str, int], List[str]]:
    """
    Build vocabulary from Word2Vec's learned keys.
    Adds <pad> and <unk>.
    """
    try:
        itos = ["<pad>", "<unk>"]
        for w in w2v_model.wv.index_to_key:
            itos.append(w)
        stoi = {w: i for i, w in enumerate(itos)}
        return stoi, itos
    except Exception as e:
        raise RuntimeError(f"Vocab creation failed: {e}") from e


def make_embedding_matrix(w2v_model, stoi: Dict[str, int], emb_dim: int) -> np.ndarray:
    """
    Create an embedding matrix aligned with stoi.

    Row 0: <pad> = zeros
    Row 1: <unk> = random small vector
    Rows 2..: Word2Vec vectors
    """
    try:
        rng = np.random.default_rng(0)
        mat = np.zeros((len(stoi), emb_dim), dtype=np.float32)

        # <unk>
        mat[stoi["<unk>"]] = rng.normal(loc=0.0, scale=0.05, size=(emb_dim,)).astype(np.float32)

        for word, idx in stoi.items():
            if word in ("<pad>", "<unk>"):
                continue
            if word in w2v_model.wv:
                mat[idx] = w2v_model.wv[word].astype(np.float32)
            else:
                # If something slips through, treat as <unk>-like.
                mat[idx] = rng.normal(loc=0.0, scale=0.05, size=(emb_dim,)).astype(np.float32)

        return mat
    except Exception as e:
        raise RuntimeError(f"Embedding matrix creation failed: {e}") from e

## Sequences

In [ ]:
def flatten_with_sentence_boundaries(sentences_tokens: List[List[str]]) -> List[str]:
    """
    Flatten sentences into one stream, inserting an <eos> token between sentences.
    This is useful so the model learns when a sentence ends.
    """
    try:
        eos = "<eos>"
        stream: List[str] = []
        for s in sentences_tokens:
            stream.extend(s)
            stream.append(eos)
        return stream
    except Exception as e:
        raise RuntimeError(f"Flatten failed: {e}") from e


def encode_stream(stream: List[str], stoi: Dict[str, int]) -> List[int]:
    """
    Encode tokens into ids, using <unk> for unseen words. Ensures <eos> exists.
    """
    try:
        if "<eos>" not in stoi:
            raise ValueError("Vocabulary is missing <eos>. Add it to your Word2Vec training data or vocab.")
        unk_id = stoi["<unk>"]
        ids: List[int] = []
        for t in stream:
            ids.append(stoi.get(t, unk_id))
        return ids
    except Exception as e:
        raise RuntimeError(f"Encoding failed: {e}") from e


def make_lm_pairs(ids: List[int], seq_len: int) -> List[Tuple[List[int], List[int]]]:
    """
    Create (x, y) for next-token prediction.
    x: [w_t ... w_{t+seq_len-1}]
    y: [w_{t+1} ... w_{t+seq_len}]
    """
    try:
        pairs: List[Tuple[List[int], List[int]]] = []
        if len(ids) <= seq_len:
            return pairs
        for i in range(0, len(ids) - seq_len):
            x = ids[i : i + seq_len]
            y = ids[i + 1 : i + seq_len + 1]
            pairs.append((x, y))
        return pairs
    except Exception as e:
        raise RuntimeError(f"Pair creation failed: {e}") from e


@dataclass
class Batch:
    x: torch.Tensor  # (B,T)
    y: torch.Tensor  # (B,T)


def sample_batch(pairs: List[Tuple[List[int], List[int]]], batch_size: int, device: str) -> Batch:
    """
    Random mini-batch sampling.
    """
    try:
        chosen = random.sample(pairs, k=batch_size)
        x = torch.tensor([p[0] for p in chosen], dtype=torch.long, device=device)
        y = torch.tensor([p[1] for p in chosen], dtype=torch.long, device=device)
        return Batch(x=x, y=y)
    except Exception as e:
        raise RuntimeError(f"Batch sampling failed: {e}") from e

## Elmann RNN

In [ ]:
class ElmanRNNLM(nn.Module):
    """
    Elman RNN language model:
      h_t = tanh(W_hh h_{t-1} + W_ih e_t + b_h)
      logits_t = W_ho h_t + b_o

    Embeddings are initialized from Word2Vec.
    """
    def __init__(
        self,
        embedding_matrix: np.ndarray,
        hidden_dim: int = 128,
        freeze_embeddings: bool = False,
    ):
        super().__init__()
        vocab_size, emb_dim = embedding_matrix.shape
        self.vocab_size = vocab_size
        self.emb_dim = emb_dim
        self.hidden_dim = hidden_dim

        self.embed = nn.Embedding(vocab_size, emb_dim)
        self.embed.weight.data = torch.tensor(embedding_matrix, dtype=torch.float32)
        self.embed.weight.requires_grad = not freeze_embeddings

        self.W_hh = nn.Parameter(torch.randn(hidden_dim, hidden_dim) * 0.1)
        self.W_ih = nn.Parameter(torch.randn(hidden_dim, emb_dim) * 0.1)
        self.b_h = nn.Parameter(torch.zeros(hidden_dim))

        self.W_ho = nn.Parameter(torch.randn(vocab_size, hidden_dim) * 0.1)
        self.b_o = nn.Parameter(torch.zeros(vocab_size))

    def forward(self, x: torch.Tensor, h0: Optional[torch.Tensor] = None) -> torch.Tensor:
        """
        x: (B,T) token ids
        returns logits: (B,T,V)
        """
        B, T = x.shape
        e = self.embed(x)  # (B,T,E)

        if h0 is None:
            h = torch.zeros(B, self.hidden_dim, device=x.device)
        else:
            h = h0

        logits_steps: List[torch.Tensor] = []
        for t in range(T):
            e_t = e[:, t, :]
            a_t = h @ self.W_hh.T + e_t @ self.W_ih.T + self.b_h
            h = torch.tanh(a_t)
            logits_t = h @ self.W_ho.T + self.b_o
            logits_steps.append(logits_t)

        return torch.stack(logits_steps, dim=1)


## Training

In [ ]:
def train_elman_lm_with_spacy_w2v(
    text: str,
    spacy_model: str = "en_core_web_sm",
    w2v_dim: int = 50,
    w2v_epochs: int = 80,
    seq_len: int = 8,
    hidden_dim: int = 128,
    batch_size: int = 32,
    steps: int = 600,
    lr: float = 1e-2,
    freeze_embeddings: bool = False,
    seed: int = 0,
):
    """
    Train pipeline:
      1) tokenize with spaCy -> sentences_tokens
      2) train Word2Vec on sentences_tokens (+ <eos> sentinel)
      3) build vocab, embedding matrix
      4) build next-token dataset
      5) train Elman LM with cross-entropy

    Returns:
      model, stoi, itos
    """
    try:
        random.seed(seed)
        torch.manual_seed(seed)
        np.random.seed(seed)

        device = "cuda" if torch.cuda.is_available() else "cpu"

        nlp = load_spacy(spacy_model)

        sentences_tokens = spacy_tokenize_paragraphs(nlp, text)

        sentences_for_w2v = []
        for s in sentences_tokens:
            sentences_for_w2v.append(s + ["<eos>"])

        w2v = train_word2vec(
            sentences_for_w2v,
            vector_size=w2v_dim,
            window=5,
            min_count=1,
            epochs=w2v_epochs,
            seed=seed,
        )

        stoi, itos = build_vocab_from_w2v(w2v)

        if "<eos>" not in stoi:
            raise ValueError("Unexpected: <eos> is missing from vocab. Check Word2Vec training input.")

        emb_matrix = make_embedding_matrix(w2v, stoi, w2v_dim)

        stream = flatten_with_sentence_boundaries(sentences_tokens)
        ids = encode_stream(stream, stoi)
        pairs = make_lm_pairs(ids, seq_len)

        if len(pairs) < batch_size:
            raise ValueError(
                f"Not enough training pairs ({len(pairs)}) for batch_size={batch_size}. "
                f"Provide more text or reduce seq_len/batch_size."
            )

        model = ElmanRNNLM(embedding_matrix=emb_matrix, hidden_dim=hidden_dim, freeze_embeddings=freeze_embeddings)
        model.to(device)

        opt = torch.optim.Adam(model.parameters(), lr=lr)

        model.train()
        for step in range(1, steps + 1):
            batch = sample_batch(pairs, batch_size=batch_size, device=device)

            opt.zero_grad()
            logits = model(batch.x)  # (B,T,V)
            B, T, V = logits.shape

            loss = F.cross_entropy(logits.reshape(B * T, V), batch.y.reshape(B * T))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            opt.step()

            if step % 100 == 0:
                print(f"step {step:4d} | loss {loss.item():.4f}")

        return model, stoi, itos

    except Exception as e:
        raise RuntimeError(f"Training pipeline failed: {e}") from e


## Usage

In [ ]:
if __name__ == "__main__":
    text = """
    Transformers are powerful sequence models. They learn contextual representations of tokens and
    capture dependencies across a sentence. A language model predicts the next token given the
    previous tokens, and training maximizes the likelihood of the observed text.
    """

    model, stoi, itos = train_elman_lm_with_spacy_w2v(
        text=text,
        w2v_dim=40,
        w2v_epochs=120,
        seq_len=6,
        hidden_dim=96,
        batch_size=16,
        steps=400,
        lr=1e-2,
        freeze_embeddings=False,  # set True to keep Word2Vec fixed
    )

step  100 | loss 0.0705
step  200 | loss 0.0825
step  300 | loss 0.0455
step  400 | loss 0.0625


# Activities

1. The code above just trains a language model, and we only see the loss function value after a number of epochs. Your first task is to show in screen this:

 * The end hidden state
 * The vector of probabilities after each 100 epochs

2. After the training loop we have that: a language model, but we have not used it for anything. Code an application, which can be one of the following:

* Score text: compute how likely a sentence/paragraph is (log-probability, perplexity).
* Generate text: start from a prompt and sample the next tokens from the language model (this is the previous stage to generate a chatbot from scratch)